# レッスン18（続き）：<em>人間</em> が操作を承認したことを証明するレシート

このレッスンは<strong>エージェント</strong>が何をしたか、そして<strong>ゲート</strong>が何を決定したかを証明します。このノートブックは不足していたもう半分、すなわち<strong>特定の人間</strong>が<strong>正確な</strong>操作を承認した証拠—完全な正準操作に対する別の、人間が保持する署名をオフラインで検証するもの—を追加します。

ここで扱う両方の成果物は、<strong>レッスンのレシートと同じエンベロープ形式</strong>を使用しています：`type` フィールドを持つ平坦なペイロードで、正準JCSバイトに対してEd25519署名が直接付与され、構造化された `signature` オブジェクトが添付（署名バイトからは除外）されています。承認レシートは新しい`type`（`human.approval.v1`）で、アクションタイプと並列で存在し、1つの`verify_chain`がメインノートブックで構築した同じコード経路で両方の成果物タイプをカバーします。この人間承認レシートは教育的な構成であり、draft-farley-acta-signed-receiptsで定義されたレシートタイプではありません。

メインノートブックのデモ検証者に対する明確な改善点の1つは、ここでの検証者が `signature.key_id` をレシート内の公開鍵を信頼せず、<strong>固定された鍵レジストリ</strong>に照らして解決することです。これはレッスン自身のチェックリストが推奨する本番環境の態勢（「検証用公開鍵を公開する」）であり、なりすましを拒否処理にし、自己鍵持ち込みによる迂回を防ぐものです。

このノートブックが示すルール：**署名された承認だけでは権限を意味しません。** 権限が成立するのは、承認レシートと操作レシートが実行時に同じ正準操作にまだ結びついていて、かつ有効なポリシーや鍵・有効期限の下にあり、かつ未使用の承認が存在するときのみです。すべての失敗は<strong>異なる理由</strong>で拒否されるため、<em>権限が期限切れ</em>なのか、<em>実行された操作が変更された</em>のかを区別できます。


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## 正確なアクション

承認の単位は<strong>正規のアクションオブジェクト</strong>です。曖昧な「返金を承認する」といったラベルではなく、正確で完全に指定されたアクションです。オブジェクト全体に署名をし（そこからダイジェストを導出し）、これによって後で人間が<em>これ</em>のみを承認したことを証明できます。


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## 1つのエンベロープ、2つの権限

すべてのレシートはレッスンのエンベロープです：`type` フィールドを持つ平坦なペイロードと、署名バイトの一部でない `signature` オブジェクト（`alg`、`sig`、`key_id`）があります。`verify_envelope` は両方のレシート種類に共通の構造＋署名検証であり、どの <strong>固定キーリポジトリ</strong> が `signature.key_id` を検証するかで権限を分けています：

- <strong>承認レシート</strong>（`human.approval.v1`） — 承認者の名前、完全な正規化アクション <strong>とそのダイジェスト</strong>、`policy_version`、発行および有効期限のタイムスタンプ。チェーンレベルでの一度だけの消費が追跡されます。
- <strong>アクションレシート</strong>（`agent.action.v1`） — エージェントの身元、`run_id`、同じ正規化アクション <strong>ダイジェスト</strong>、実行結果＋タイムスタンプ、および `parent_approval_ref`：承認の `receipt_hash` で、レッスンのチェーン内の `previous_receipt_hash` と同じ慣習を持ちます。

共有された `action_digest` フィールドがバインディングの結合点です。`key_id` は署名オブジェクト内に参照ヒントとして存在するだけで、異なる固定キーに向けると署名検証が失敗するため、何も提供しません。


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: バインディングが実際に決定される場所

`verify_chain` は2つの署名チェックの単なる便宜的なラッパーではありません。共有された正準の `action_digest`、承認のポリシー/鍵/有効期限の<strong>新鮮さ</strong>、および承認の<strong>一回限りの消費</strong>が、実行中のアクションに対して<em>同時に</em>チェックされる唯一の場所です。

各失敗は<strong>異なる理由</strong>で拒否されるため、拒否を読む者は権限が古くなったのか（ポリシーの変更、鍵のローテーション、承認の期限切れ、承認の消費）それとも有効な承認の下で実行されるアクションが変わったのか（ダイジェストの置換）を判断できます。


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## バインディングが捕捉するもの

以下の各ケースはすべて、<strong>異なる理由で</strong> **closed** に失敗します。最初のブロックはクラシックなセット（改ざん、混乱した代理、リプレイ、いずれかの権限による偽造、形式不正な入力）です。二番目のブロックは、プロパティを単なる主張ではなく実際のものにするペアです：

- <strong>古い権限</strong> — 署名はまだ有効ですが、ポリシーのバージョンが変わった、承認者キーがピン留めレジストリからローテーションされた、または実行前に承認が期限切れになった場合；
- <strong>ダイジェスト差替え</strong> — 実際の承認を指す `parent_approval_ref` を持つ有効に署名されたアクション受付通知ですが、その承認の標準的なアクションダイジェストが実際に実行されているアクションと一致しない場合。


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## これが証明するもの — そして証明しないもの

**証明すること:** 名前付きの人間が<em>この正確な標準的なアクション</em>（完全なアクション＋ダイジェスト、ピン留めされたレジストリから解決されたキーで署名されている）を承認し、エージェントが<em>そのままの承認されたアクション</em>（同じダイジェスト、承認に `receipt_hash` で紐づけられたレシート、レッスン独自のチェーン規約）を正確に一度だけ実行したこと。どちらかが変わるとチェーンは閉じて失敗し、拒否理由が<strong>どの</strong>プロパティが壊れたか（古い権限か変更されたアクションか）を教える。

**証明しないこと:** 承認UIが人間に実際に署名すると考えたものを表示したかどうか（WYSIWYSは別の問題）、キーが回転前に強制されたり盗まれたりしていないこと、または下流の効果がアクションと一致していること。署名＝認可ではない: 古いポリシー、有効期限切れのウィンドウ、異なるダイジェスト、または回転されたキー上の有効な署名はここでは何も保証しない。

２種類のレシートはレッスンのエンベロープと１つの `verify_chain` コードパスを意図的に共有している: メインのノートブックでアクションレシート用に構築したバインディングが、人間の承認を検証するのと同じコードだということ。１つの検証契約、別々のピン留めされた権限、標準的なアクションのダイジェストのみで結合されている。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
